# Pipelines that return images

[Back to the introduction](ch01_demo.ipynb). Run in order. Four `LLM.pipeline()` tasks answer with images: background removal, depth estimation, image segmentation and image-to-image (here, a 2x upscale). Each image comes back as PNG bytes that `IPython.display.Image(data=...)` shows; pass `as_array=True` for a `uint8` NumPy array of shape (height, width, channels) instead. Each image crosses from the browser as one buffer, and the PNG is encoded in the browser, so no Pillow is needed. All four models here are small and released under Apache-2.0. Each section loads a model, runs it on the same photo and releases it.

In [ ]:
import struct

from IPython.display import Image, display

from lmlite import LLM

url = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/tiger.jpg"


def png_size(png):
    """(width, height) from a PNG's header."""
    return struct.unpack(">II", png[16:24])


display(Image(url=url, width=360))

## Background removal

`modnet` cuts out the foreground. The result is an RGBA PNG: the background is transparent, so it shows the page behind it.

In [ ]:
remover = await LLM.pipeline("background-removal", model="modnet", backend="wasm")
cutout = await remover(url)
print(png_size(cutout), "pixels,", len(cutout), "bytes")
display(Image(data=cutout, width=360))
await remover.dispose()

## Depth estimation

Depth Anything returns a depth image (brighter is closer) and `predicted_depth`, the model's raw values as a float32 NumPy array of shape (height, width), or as nested lists when NumPy is not installed.

In [ ]:
estimator = await LLM.pipeline("depth-estimation", model="onnx-community/depth-anything-v2-small", backend="wasm")
depth = await estimator(url)
values = depth["predicted_depth"]
print(png_size(depth["depth"]), "pixels; raw values", getattr(values, "shape", None) or (len(values), len(values[0])))
display(Image(data=depth["depth"], width=360))
await estimator.dispose()

## Image segmentation

DETR (panoptic) splits a photo into segments. Each is a dict with a `label`, a `score` (`None` for models that do not give one) and a `mask`: a PNG that is white where the segment is. This section uses a photo of two cats, because the tiger is no object class the model knows.

Two things to know about this model. Its quantized default file (`q8`, 44 MB) returns masks that are wrong, for example one segment covering the whole photo, and the half-precision file (`fp16`) can run out of memory in a fresh page (`std::bad_alloc`), so the notebook asks for `dtype="fp32"` (172 MB). And its config names the object classes (`cat`, `couch`, `remote`) but not the background ones, which come back as `LABEL_<id>`.

In [ ]:
cats = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/cats.jpg"
display(Image(url=cats, width=300))

segmenter = await LLM.pipeline("image-segmentation", model="detr-resnet-50-panoptic", dtype="fp32", backend="wasm")
segments = await segmenter(cats)
for segment in segments:
    score = segment["score"]  # None for models that give no score
    print(segment["label"], round(score, 3) if score is not None else "no score")
    display(Image(data=segment["mask"], width=240))
await segmenter.dispose()

## Image to image: a 2x upscale

Swin2SR doubles the width and height. It pads the photo to a multiple of its window, so the result can be a few pixels larger than exactly twice the input.

In [ ]:
upscaler = await LLM.pipeline("image-to-image", model="swin2SR-lightweight-x2-64", backend="wasm")
larger = await upscaler(url)
print("upscaled to", png_size(larger), "pixels,", len(larger), "bytes")
display(Image(data=larger, width=360))
await upscaler.dispose()

The inference pipelines are released; cached files are retained for subsequent runs. `await pipe.map(images)` runs any of these over many photos with progress, and the tested models and licences are in the model guide.